In [9]:
# If you're on Python < 3.10, add this first line so `int | None` in type hints parses:
from __future__ import annotations

import math
from typing import Any, Dict, List, Tuple

import numpy as np

# PyTorch (needed for the dataset, model, training, and plotting predictions)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Plotting (3D scatter + color normalization)
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  # ensures 3D projection is registered


In [10]:
# ===== kNN on arbitrary coordinates (Euclidean) =====
def build_knn_graph_euclidean(X: np.ndarray, k: int = 16):
    """Symmetric kNN with Gaussian weights on Euclidean distances.
       Return W_raw, Wf, neighbors, deg_unw, sigma^2."""
    N = X.shape[0]
    sq = np.sum(X*X, axis=1)
    XXT = X @ X.T
    D2 = sq[:,None] + sq[None,:] - 2.0*XXT
    D2 = np.clip(D2, 0.0, None)
    np.fill_diagonal(D2, np.inf)

    kth_sq = np.partition(D2, k, axis=1)[:, k]
    sigma2 = float(np.median(kth_sq))

    W = np.zeros((N, N), dtype=float)
    for i in range(N):
        nn = np.argpartition(D2[i], k)[:k]
        W[i, nn] = np.exp(-D2[i, nn] / sigma2)
    W = np.maximum(W, W.T)
    np.fill_diagonal(W, 0.0)
    deg = W.sum(axis=1)
    D_inv_sqrt = 1.0 / np.sqrt(deg + 1e-12)
    Wf = (D_inv_sqrt[:, None] * W) * D_inv_sqrt[None, :]

    A_unw = (W > 0).astype(np.int32)
    deg_unw = A_unw.sum(axis=1).astype(np.int32)
    neighbors = [np.flatnonzero(A_unw[i]).astype(np.int32) for i in range(N)]
    return W, Wf, neighbors, deg_unw, sigma2

# ===== Signatures from arbitrary coordinates (same walk estimator you wrote) =====
def build_ggrf_signatures_from_coords(
    X: np.ndarray, k: int, num_starts: int, start_mode: str,
    n_walks: int | None, phalt: float, tau: float, seed: int
) -> Dict[str, Any]:
    rng = np.random.default_rng(seed)
    W_raw, Wf, neighbors, deg_unw, sigma2 = build_knn_graph_euclidean(X, k=k)

    N = X.shape[0]
    if start_mode == "farthest":
        start_indices = farthest_point_sample(X, num_starts, seed=seed)
    elif start_mode == "random":
        start_indices = rng.choice(N, size=num_starts, replace=False)
    else:
        raise ValueError("start_mode must be 'farthest' or 'random'")

    alpha = precompute_alpha(tau, Kmax=10000, eps=1e-300)
    def modulation(step: int) -> float:
        return float(alpha[step]) if step < alpha.shape[0] else 0.0

    if n_walks is None: n_walks = N
    phi = np.zeros((num_starts, N), dtype=float)

    def sample_feature(start_idx: int):
        vec = np.zeros(N, dtype=float)
        local_rng = np.random.default_rng(int(seed) * 1009 + int(start_idx))
        for _ in range(n_walks):
            u = int(start_idx)
            load = 1.0
            step = 0
            while True:
                vec[u] += load * modulation(step)
                step += 1
                nbrs = neighbors[u]
                if nbrs.size == 0: break
                v = int(nbrs[local_rng.integers(0, nbrs.size)])
                # uniform-by-count neighbor sampling correction + geometric continuation correction
                load *= (deg_unw[u] / (1.0 - phalt)) * float(Wf[u, v])
                u = v
                # terminate after the transition with prob phalt
                if local_rng.random() < phalt:
                    break
        vec /= float(n_walks)
        return vec

    for j, s in enumerate(start_indices):
        phi[j] = sample_feature(int(s))

    return {
        "X": X,
        "start_indices": start_indices,
        "starts_xyz": X[start_indices],
        "phi": phi,
        "params": {
            "N": int(N), "k": int(k), "num_starts": int(num_starts),
            "start_mode": start_mode, "n_walks": int(n_walks),
            "phalt": float(phalt), "tau": float(tau), "sigma2": float(sigma2)
        },
        "Wf": Wf, "neighbors": neighbors
    }

# ===== Geodesics =====
def geodesic_matrix_sphere(X: np.ndarray) -> np.ndarray:
    """Exact spherical geodesic on S^2 via arccos of dot product."""
    dots = np.clip(X @ X.T, -1.0, 1.0)
    return np.arccos(dots)

import heapq
def dijkstra_single_source(X: np.ndarray, neighbors: List[np.ndarray], s: int) -> np.ndarray:
    """Geodesic (graph-shortest paths) from source s using Euclidean edge lengths."""
    N = X.shape[0]
    dist = np.full(N, np.inf, dtype=float)
    dist[s] = 0.0
    h = [(0.0, s)]
    while h:
        d,u = heapq.heappop(h)
        if d>dist[u]: continue
        Xu = X[u]
        for v in neighbors[u]:
            w = float(np.linalg.norm(Xu - X[int(v)]))
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                heapq.heappush(h, (nd, int(v)))
    return dist

def geodesic_matrix_graph(X: np.ndarray, neighbors: List[np.ndarray], sources: np.ndarray) -> np.ndarray:
    """Compute geodesic distances (rows only for given sources) via Dijkstra on kNN graph."""
    N = X.shape[0]
    G = np.zeros((len(sources), N), dtype=float)
    for i, s in enumerate(sources):
        G[i] = dijkstra_single_source(X, neighbors, int(s))
    return G


In [11]:
# ===== start-wise split =====
def split_by_start(starts_all: np.ndarray, val_frac: float = 0.2, seed: int = 0):
    rng = np.random.default_rng(seed)
    unique = np.unique(starts_all)
    rng.shuffle(unique)
    m_val = max(1, int(round(len(unique) * val_frac)))
    val_starts = np.sort(unique[:m_val])
    train_starts = np.sort(unique[m_val:])
    train_mask = np.isin(starts_all, train_starts)
    val_mask   = np.isin(starts_all, val_starts)
    train_idx = np.nonzero(train_mask)[0]
    val_idx   = np.nonzero(val_mask)[0]
    return train_idx, val_idx, train_starts, val_starts

# ===== dataset that uses a precomputed geodesic matrix =====
if ensure_torch():
    class NodePairDatasetWithGeod(NodePairDataset):
        def __init__(self, sup: Dict[str, Any], geod_matrix: np.ndarray, use_continuous_only: bool = True):
            super().__init__(sup, use_continuous_only=use_continuous_only)
            # geod_matrix can be full N x N, or rows only for the starts (pass an indexer to map)
            self.G = geod_matrix  # shape (N, N) OR (num_starts, N) if you index by start position
            self.map_from_start_to_row = None  # filled if G is subset

        def set_row_indexer(self, start_indices: np.ndarray):
            """Optional: if G has rows only for specific start nodes in 'start_indices' (validation/train starts),
               set this so we can pick correct geodesic rows quickly."""
            pos = {int(s): i for i, s in enumerate(start_indices)}
            self.map_from_start_to_row = pos

        def __getitem__(self, idx):
            ex = super().__getitem__(idx)
            s = int(ex["start"]); o = int(ex["omega"])
            if self.map_from_start_to_row is None:
                geod_val = float(self.G[s, o])
            else:
                geod_val = float(self.G[self.map_from_start_to_row[s], o])
            ex["geod"] = geod_val
            return ex

    # ===== training with start-wise split + history =====
    def train_model_node_pairs_by_start(
        sup: Dict[str, Any], geod_matrix: np.ndarray,
        batch_size=1024, epochs=8, lr=1e-3,
        d_emb=64, hidden=128, use_continuous_only=True,
        val_frac=0.2, seed=0, val_start_ids: np.ndarray | None = None
    ):
        import torch
        ds = NodePairDatasetWithGeod(sup, geod_matrix, use_continuous_only=use_continuous_only)

        if val_start_ids is None:
            tr_idx, va_idx, tr_starts, va_starts = split_by_start(sup["starts"], val_frac=val_frac, seed=seed)
        else:
            # user-specified validation starts
            all_starts = np.unique(sup["starts"])
            tr_starts = np.setdiff1d(all_starts, val_start_ids)
            va_starts = np.array(sorted(np.unique(val_start_ids)))
            tr_idx = np.nonzero(np.isin(sup["starts"], tr_starts))[0]
            va_idx = np.nonzero(np.isin(sup["starts"], va_starts))[0]

        class Subset(torch.utils.data.Dataset):
            def __init__(self, base, idxs): self.base, self.idxs = base, idxs
            def __len__(self): return len(self.idxs)
            def __getitem__(self, i): return self.base[self.idxs[i]]

        tr_ds, va_ds = Subset(ds, tr_idx), Subset(ds, va_idx)
        tr = torch.utils.data.DataLoader(tr_ds, batch_size=batch_size, shuffle=True, collate_fn=collate, drop_last=False)
        va = torch.utils.data.DataLoader(va_ds, batch_size=batch_size, shuffle=False, collate_fn=collate, drop_last=False)

        model = SignatureAtNodeRegressor(
            num_nodes=sup["N"], x_coords=sup["X"],
            d_emb=d_emb, hidden=hidden, use_continuous_only=use_continuous_only
        )
        torch.manual_seed(seed)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = torch.nn.MSELoss()

        def eval_loader(dl):
            model.eval()
            se, n = 0.0, 0
            with torch.no_grad():
                for batch in dl:
                    pred = model(batch); y = batch["target"]
                    se += torch.sum((pred - y)**2).item()
                    n += y.numel()
            return math.sqrt(se / n)

        hist = {"rmse_train": [], "rmse_val": []}
        for ep in range(1, epochs+1):
            model.train()
            for batch in tr:
                pred = model(batch); y = batch["target"]
                loss = loss_fn(pred, y)
                opt.zero_grad(); loss.backward(); opt.step()
            rmse_tr = eval_loader(tr)
            rmse_va = eval_loader(va)
            hist["rmse_train"].append(rmse_tr)
            hist["rmse_val"].append(rmse_va)
            print(f"Epoch {ep:02d} | RMSE train: {rmse_tr:.6f} | RMSE val: {rmse_va:.6f}")

        return model, hist, {"train_starts": tr_starts, "val_starts": va_starts,
                             "train_idx": tr_idx, "val_idx": va_idx}

# ===== training/validation dynamics plot =====
def plot_dynamics(hist, title="Training/Validation RMSE"):
    import matplotlib.pyplot as plt
    xs = np.arange(1, len(hist["rmse_train"])+1)
    plt.figure()
    plt.plot(xs, hist["rmse_train"], label="train")
    plt.plot(xs, hist["rmse_val"],   label="val")
    plt.xlabel("epoch"); plt.ylabel("RMSE"); plt.title(title)
    plt.legend(); plt.show()


NameError: name 'ensure_torch' is not defined

In [4]:
def plot_fx_field_for_start_generic(model, X, start_indices, phi, s_idx, geod_vec=None,
                                    title_prefix="", share_norm=True,
                                    cmap_main="OrRd", cmap_err="coolwarm"):
    """Same as your plot_fx_field, but you can pass geod_vec explicitly (for non-sphere cases)."""
    import matplotlib.pyplot as plt
    from matplotlib.colors import Normalize, TwoSlopeNorm
    from matplotlib.cm import ScalarMappable
    model.eval()
    device = next(model.parameters()).device
    N = X.shape[0]
    s = int(start_indices[s_idx])
    start_xyz = X[s]
    omega_xyz = X

    if geod_vec is None:
        dots = np.clip((omega_xyz * start_xyz).sum(axis=1), -1.0, 1.0)
        geod = np.arccos(dots)
    else:
        geod = geod_vec.astype(np.float32)

    batch = {
        "start": torch.full((N,), s, dtype=torch.long, device=device),
        "omega": torch.arange(N, dtype=torch.long, device=device),
        "start_xyz": torch.tensor(np.repeat(start_xyz[None,:], N, axis=0), dtype=torch.float32, device=device),
        "omega_xyz": torch.tensor(omega_xyz, dtype=torch.float32, device=device),
        "geod": torch.tensor(geod[:,None], dtype=torch.float32, device=device),
    }
    with torch.no_grad():
        f_pred = model(batch).detach().cpu().numpy()

    # truth row for this start
    j = int(np.where(start_indices == s)[0][0])
    phi_row = phi[j].astype(float)

    # shared or independent normalization
    if share_norm:
        vmin = min(f_pred.min(), phi_row.min())
        vmax = max(f_pred.max(), phi_row.max())
        norm_main = Normalize(vmin=vmin, vmax=vmax)
        f_vis = f_pred; g_vis = phi_row
    else:
        f_vis = (f_pred - f_pred.min()) / (f_pred.max() - f_pred.min() + 1e-12)
        g_vis = (phi_row - phi_row.min()) / (phi_row.max() - phi_row.min() + 1e-12)
        norm_main = Normalize(vmin=0.0, vmax=1.0)

    err = f_pred - phi_row
    norm_err = TwoSlopeNorm(vmin=err.min(), vcenter=0.0, vmax=err.max())

    # ---- PRED ----
    fig1 = plt.figure()
    ax1 = fig1.add_subplot(111, projection='3d')
    ax1.scatter(X[:,0], X[:,1], X[:,2], s=18, c=f_vis, cmap=cmap_main, norm=norm_main)
    ax1.set_title(f"{title_prefix} NN prediction f(x, ·)")
    ax1.set_xticks([]); ax1.set_yticks([]); ax1.set_zticks([]); ax1.set_box_aspect([1,1,1])
    cbar1 = fig1.colorbar(ScalarMappable(norm=norm_main, cmap=cmap_main), ax=ax1, shrink=0.8)
    cbar1.set_label("f(x, ω)" + (" (shared scale)" if share_norm else " (individually normalized)"))
    plt.show()

    # ---- TRUTH ----
    fig2 = plt.figure()
    ax2 = fig2.add_subplot(111, projection='3d')
    ax2.scatter(X[:,0], X[:,1], X[:,2], s=18, c=g_vis, cmap=cmap_main, norm=norm_main)
    ax2.set_title(f"{title_prefix} Ground truth φ_t(x)[·]")
    ax2.set_xticks([]); ax2.set_yticks([]); ax2.set_zticks([]); ax2.set_box_aspect([1,1,1])
    cbar2 = fig2.colorbar(ScalarMappable(norm=norm_main, cmap=cmap_main), ax=ax2, shrink=0.8)
    cbar2.set_label("φ_t(x, ω)" + (" (shared scale)" if share_norm else " (individually normalized)"))
    plt.show()

    # ---- ERROR ----
    fig3 = plt.figure()
    ax3 = fig3.add_subplot(111, projection='3d')
    ax3.scatter(X[:,0], X[:,1], X[:,2], s=18, c=err, cmap=cmap_err, norm=norm_err)
    ax3.set_title(f"{title_prefix} Error: f(x, ·) − φ_t(x)[·]")
    ax3.set_xticks([]); ax3.set_yticks([]); ax3.set_zticks([]); ax3.set_box_aspect([1,1,1])
    cbar3 = fig3.colorbar(ScalarMappable(norm=norm_err, cmap=cmap_err), ax=ax3, shrink=0.8)
    cbar3.set_label("Prediction − Truth")
    plt.show()

def visualize_several_validation_starts(model, X, start_indices, phi, val_start_ids, how_many=3,
                                        geod_rows: Dict[int, np.ndarray] | None = None,
                                        title_prefix="Val start"):
    """Pick first `how_many` validation start nodes and plot."""
    val_list = list(val_start_ids)[:how_many]
    for s in val_list:
        j = int(np.where(start_indices == s)[0][0])
        geod_vec = None if geod_rows is None else geod_rows[int(s)]
        plot_fx_field_for_start_generic(model, X, start_indices, phi, j,
                                        geod_vec=geod_vec,
                                        title_prefix=f"{title_prefix} s={int(s)} | j={j} ")


NameError: name 'Dict' is not defined

In [5]:
# ===== manifolds =====
def fibonacci_sphere_scaled(n: int, a=1.0, b=1.0, c=1.0) -> np.ndarray:
    """Ellipsoid points by scaling the unit-sphere Fibonacci sample with axes a,b,c."""
    S = fibonacci_sphere(n)
    E = S * np.array([a,b,c])[None,:]
    return E

def swiss_roll(n: int, noise: float = 0.0, seed: int = 0) -> np.ndarray:
    """Swiss roll in R^3."""
    rng = np.random.default_rng(seed)
    t = 1.5 * np.pi * (1 + 2 * rng.random(n))
    h = 21 * rng.random(n)
    x = t * np.cos(t)
    y = h
    z = t * np.sin(t)
    X = np.stack([x,y,z], axis=1)
    if noise > 0.0:
        X = X + noise * rng.standard_normal(size=X.shape)
    # center/scale for nicer plots
    X = X / np.std(X, axis=0)
    return X


In [6]:
# Build signatures on S^2 (your code does this already)
out = build_ggrf_signatures_on_sphere(
    N=CFG["N"], k=CFG["k"], num_starts=CFG["num_starts"],
    start_mode=CFG["start_mode"], n_walks=CFG["n_walks"],
    phalt=CFG["phalt"], tau=CFG["t"], seed=CFG["seed"]
)
X, phi, start_indices = out["X"], out["phi"], out["start_indices"]

# Supervision (all ω per start or subsample)
sup = make_node_pair_supervision(
    out,
    mode=("all" if CFG["subsample_per_start"] is None else "random"),
    M_per_start=CFG["subsample_per_start"],
    seed=123
)
sup["N"] = out["params"]["N"]

# Geodesics on the sphere
G_sphere = geodesic_matrix_sphere(X)  # N x N

# Train with start-wise holdout (so any visualized row is validation-only)
model_sphere, hist_sphere, splits_sphere = train_model_node_pairs_by_start(
    sup, G_sphere,
    batch_size=CFG["batch_size"], epochs=CFG["epochs"], lr=CFG["lr"],
    d_emb=CFG["d_emb"], hidden=CFG["hidden"], use_continuous_only=CFG["use_continuous_only"],
    val_frac=CFG["val_frac"], seed=CFG["torch_seed"]
)

plot_dynamics(hist_sphere, title="Sphere: RMSE")

# Visualize several validation starts (e.g., 3)
visualize_several_validation_starts(
    model_sphere, X, start_indices, phi,
    val_start_ids=splits_sphere["val_starts"], how_many=3,
    geod_rows=None,  # we compute spherical geodesic on the fly in the plotting function
    title_prefix="Sphere | val"
)


NameError: name 'build_ggrf_signatures_on_sphere' is not defined

In [7]:
# Build ellipsoid points
X_ell = fibonacci_sphere_scaled(CFG["N"], a=1.0, b=1.3, c=0.7)

# Signatures on the ellipsoid graph
out_ell = build_ggrf_signatures_from_coords(
    X_ell, k=CFG["k"], num_starts=CFG["num_starts"], start_mode=CFG["start_mode"],
    n_walks=CFG["n_walks"], phalt=CFG["phalt"], tau=CFG["t"], seed=CFG["seed"]
)
X_ell, phi_ell, start_indices_ell = out_ell["X"], out_ell["phi"], out_ell["start_indices"]

sup_ell = make_node_pair_supervision(
    {"X": X_ell, "phi": phi_ell, "start_indices": start_indices_ell,
     "params": {"N": X_ell.shape[0]}},
    mode=("all" if CFG["subsample_per_start"] is None else "random"),
    M_per_start=CFG["subsample_per_start"], seed=123
)
sup_ell["N"] = X_ell.shape[0]

# Graph geodesics (Dijkstra) from all start nodes (train+val) to all nodes
G_rows = geodesic_matrix_graph(X_ell, out_ell["neighbors"], sources=start_indices_ell)
# Assemble a full N x N geodesic matrix (only rows for starts are needed by the dataset)
G_full = np.zeros((X_ell.shape[0], X_ell.shape[0]), dtype=float)
for i, s in enumerate(start_indices_ell):
    G_full[s] = G_rows[i]

model_ell, hist_ell, splits_ell = train_model_node_pairs_by_start(
    sup_ell, G_full,
    batch_size=CFG["batch_size"], epochs=CFG["epochs"], lr=CFG["lr"],
    d_emb=CFG["d_emb"], hidden=CFG["hidden"], use_continuous_only=CFG["use_continuous_only"],
    val_frac=CFG["val_frac"], seed=CFG["torch_seed"]
)
plot_dynamics(hist_ell, title="Ellipsoid: RMSE")

# For plotting, pass geodesic rows so the MLP gets exactly the same geodesic feature we trained on
geod_rows_dict = {int(s): G_full[int(s)] for s in start_indices_ell}
visualize_several_validation_starts(
    model_ell, X_ell, start_indices_ell, phi_ell,
    val_start_ids=splits_ell["val_starts"], how_many=3,
    geod_rows=geod_rows_dict, title_prefix="Ellipsoid | val"
)


NameError: name 'CFG' is not defined

In [8]:
X_sw = swiss_roll(CFG["N"], noise=0.0, seed=CFG["seed"])

out_sw = build_ggrf_signatures_from_coords(
    X_sw, k=CFG["k"], num_starts=CFG["num_starts"], start_mode=CFG["start_mode"],
    n_walks=CFG["n_walks"], phalt=CFG["phalt"], tau=CFG["t"], seed=CFG["seed"]
)
X_sw, phi_sw, start_indices_sw = out_sw["X"], out_sw["phi"], out_sw["start_indices"]

sup_sw = make_node_pair_supervision(
    {"X": X_sw, "phi": phi_sw, "start_indices": start_indices_sw,
     "params": {"N": X_sw.shape[0]}},
    mode=("all" if CFG["subsample_per_start"] is None else "random"),
    M_per_start=CFG["subsample_per_start"], seed=123
)
sup_sw["N"] = X_sw.shape[0]

# Graph geodesics from all start nodes
G_rows_sw = geodesic_matrix_graph(X_sw, out_sw["neighbors"], sources=start_indices_sw)
G_full_sw = np.zeros((X_sw.shape[0], X_sw.shape[0]), dtype=float)
for i, s in enumerate(start_indices_sw):
    G_full_sw[s] = G_rows_sw[i]

model_sw, hist_sw, splits_sw = train_model_node_pairs_by_start(
    sup_sw, G_full_sw,
    batch_size=CFG["batch_size"], epochs=CFG["epochs"], lr=CFG["lr"],
    d_emb=CFG["d_emb"], hidden=CFG["hidden"], use_continuous_only=CFG["use_continuous_only"],
    val_frac=CFG["val_frac"], seed=CFG["torch_seed"]
)
plot_dynamics(hist_sw, title="Swiss roll: RMSE")

geod_rows_sw = {int(s): G_full_sw[int(s)] for s in start_indices_sw}
visualize_several_validation_starts(
    model_sw, X_sw, start_indices_sw, phi_sw,
    val_start_ids=splits_sw["val_starts"], how_many=3,
    geod_rows=geod_rows_sw, title_prefix="Swiss roll | val"
)


NameError: name 'CFG' is not defined